In [3]:
import sys
from pathlib import Path

# Add the project root (E:\Healthcare) to Python's import path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: e:\Healthcare


In [4]:
import pandas as pd

from ingestion.validate import (
    validate_required_columns,
    validate_required_values,
    validate_encounter_ids,
    validate_dates,
)

In [5]:
import pandas as pd

# Load the raw healthcare dataset
df = pd.read_csv("../data/raw/ehr_encounters.csv")

print("Dataset shape:", df.shape)

Dataset shape: (4177, 102)


In [6]:
# ### 1. Required Column Validation

# These columns are essential for identifying and tracking healthcare encounters.

# We need to confirm that:
# - encounter_id exists
# - patient_id exists
# - encounter_date exists

# If any of these columns are missing, the pipeline should stop before loading the data into the warehouse.

In [7]:
missing_columns = validate_required_columns(df)

print("Missing required columns:", missing_columns)

if not missing_columns:
    print("PASS: All required columns are present.")
else:
    print("FAIL: Required columns are missing.")

Missing required columns: []
PASS: All required columns are present.


In [8]:
# ### 2. Required Value Validation

# Required identifiers and dates should not be null.

# We check:
# - encounter_id
# - patient_id
# - encounter_date

# Any record failing these checks will be considered invalid.

In [9]:
invalid_required = validate_required_values(df)

print("Invalid records:", len(invalid_required))

if len(invalid_required) == 0:
    print("PASS: No missing required values.")
else:
    print("FAIL: Records contain missing required values.")
    display(invalid_required.head())

Invalid records: 0
PASS: No missing required values.


In [10]:
# ### 3. Encounter ID Uniqueness

# Each healthcare encounter should have a unique encounter_id.

# Patient IDs are allowed to repeat because one patient can have multiple healthcare encounters.

# Therefore, we validate encounter_id rather than patient_id for uniqueness.

In [11]:
invalid_encounter_ids = validate_encounter_ids(df)

print("Duplicate encounter ID records:", len(invalid_encounter_ids))

if len(invalid_encounter_ids) == 0:
    print("PASS: All encounter IDs are unique.")
else:
    print("FAIL: Duplicate encounter IDs found.")
    display(invalid_encounter_ids.head())

Duplicate encounter ID records: 0
PASS: All encounter IDs are unique.


In [12]:
# ### 4. Date Validation

# Healthcare encounter dates must be valid.

# We check:
# - encounter_date can be converted to a valid date
# - discharge_date can be converted when present
# - discharge_date should not occur before encounter_date

In [13]:
invalid_dates = validate_dates(df)

print("Invalid date records:", len(invalid_dates))

if len(invalid_dates) == 0:
    print("PASS: Date validation passed.")
else:
    print("FAIL: Invalid date records found.")
    display(invalid_dates.head())

Invalid date records: 0
PASS: Date validation passed.


In [14]:
# ### 5. Validation Summary

# The following summary records the result of each validation rule.

# This gives us a simple data-quality report that can later be incorporated into the ingestion pipeline.

In [15]:
validation_summary = pd.DataFrame({
    "validation": [
        "Required columns",
        "Required values",
        "Unique encounter IDs",
        "Valid dates"
    ],
    "invalid_records": [
        len(missing_columns),
        len(invalid_required),
        len(invalid_encounter_ids),
        len(invalid_dates)
    ]
})

validation_summary["status"] = validation_summary["invalid_records"].apply(
    lambda x: "PASS" if x == 0 else "FAIL"
)

display(validation_summary)

,validation,invalid_records,status
0,Required columns,0,PASS
1,Required values,0,PASS
2,Unique encounter IDs,0,PASS
3,Valid dates,0,PASS


In [16]:
from pathlib import Path

output_path = Path("../data/processed/validation_summary.csv")

validation_summary.to_csv(output_path, index=False)

print(f"Validation report saved to: {output_path}")

Validation report saved to: ..\data\processed\validation_summary.csv


In [17]:
# ### 6. Secondary Diagnosis Consistency

# The dataset contains a reported count of secondary diagnoses (`n_secondary_dx`)
# and up to five secondary diagnosis columns.

# We compare the reported count with the number of populated secondary diagnosis
# fields to identify inconsistent records.

In [18]:
secondary_cols = [
    "dx_secondary_1",
    "dx_secondary_2",
    "dx_secondary_3",
    "dx_secondary_4",
    "dx_secondary_5"
]

# Count how many secondary diagnosis fields are populated
calculated_secondary_count = df[secondary_cols].notna().sum(axis=1)

# Compare reported count with calculated count
secondary_check = pd.DataFrame({
    "encounter_id": df["encounter_id"],
    "reported_count": df["n_secondary_dx"],
    "calculated_count": calculated_secondary_count
})

secondary_check["matches"] = (
    secondary_check["reported_count"]
    == secondary_check["calculated_count"]
)

display(secondary_check.head())

,encounter_id,reported_count,calculated_count,matches
0,545a8301-103a-4df7-bae6-ef033cc97092,0,0,True
1,4706d33b-2f7b-4838-a7bf-bb05b90d2570,0,0,True
2,079d12ad-4328-4023-89dd-c5d4f9521da7,0,0,True
3,42b40aa5-5eb4-48af-8607-97385f5da23e,0,0,True
4,6c6589ec-4464-4fcb-851a-8e71e5707a3d,0,0,True


In [19]:
print("Total records:", len(secondary_check))
print(
    "Matching records:",
    secondary_check["matches"].sum()
)
print(
    "Mismatching records:",
    (~secondary_check["matches"]).sum()
)

Total records: 4177
Matching records: 4177
Mismatching records: 0


In [20]:
# ### 7. Length-of-Stay Consistency

# The dataset contains both encounter/discharge dates and a reported
# length-of-stay value (`los_days`).

# We calculate the date difference and compare it with the reported value.

# This helps identify records where related fields may be inconsistent.

In [21]:
# Convert dates to datetime
encounter_dates = pd.to_datetime(
    df["encounter_date"],
    errors="coerce"
)

discharge_dates = pd.to_datetime(
    df["discharge_date"],
    errors="coerce"
)

# Calculate length of stay from the dates
calculated_los = (
    discharge_dates - encounter_dates
).dt.days

# Create comparison table
los_check = pd.DataFrame({
    "encounter_id": df["encounter_id"],
    "reported_los": df["los_days"],
    "calculated_los": calculated_los
})

# Calculate difference
los_check["difference"] = (
    los_check["reported_los"]
    - los_check["calculated_los"]
)

display(los_check.head(10))

,encounter_id,reported_los,calculated_los,difference
0,545a8301-103a-4df7-bae6-ef033cc97092,0,0,0
1,4706d33b-2f7b-4838-a7bf-bb05b90d2570,0,0,0
2,079d12ad-4328-4023-89dd-c5d4f9521da7,0,0,0
3,42b40aa5-5eb4-48af-8607-97385f5da23e,0,0,0
4,6c6589ec-4464-4fcb-851a-8e71e5707a3d,0,0,0
5,49fb511b-47e4-48fd-a1d4-32cb1d3508ff,0,0,0
6,cc40a249-ca91-4e6c-a457-3ad5882ca245,0,0,0
7,243d8480-da32-4236-a768-960dd1b88f1f,0,0,0
8,457391d0-5a8f-4568-aa7f-fa8aa589d884,0,0,0
9,672abd61-842d-4a77-bfb8-37a6b0291f55,0,0,0


In [22]:
print("Total records:", len(los_check))

print(
    "Exact matches:",
    (los_check["difference"] == 0).sum()
)

print(
    "Mismatches:",
    (los_check["difference"] != 0).sum()
)

print(
    "Missing calculated LOS:",
    los_check["calculated_los"].isna().sum()
)

Total records: 4177
Exact matches: 4177
Mismatches: 0
Missing calculated LOS: 0


In [23]:
# ### 8. Numeric Data Quality Validation

# Numeric validation checks whether selected fields contain structurally invalid
# values such as negative counts or percentages outside their valid numeric range.

# These checks are data-quality rules rather than clinical assessments.

In [24]:
numeric_rules = {
    "age": lambda s: s <= 0,
    "los_days": lambda s: s < 0,
    "n_secondary_dx": lambda s: (s < 0) | (s > 5),
    "n_medications": lambda s: s < 0,
    "note_word_count": lambda s: s < 0,
    "vs_spo2_pct": lambda s: (s < 0) | (s > 100),
    "vs_height_cm": lambda s: s <= 0,
    "vs_weight_kg": lambda s: s <= 0,
    "vs_bmi": lambda s: s <= 0,
}

numeric_validation_results = []

for column, rule in numeric_rules.items():
    invalid_count = rule(df[column]).sum()

    numeric_validation_results.append({
        "column": column,
        "invalid_records": int(invalid_count),
        "status": "PASS" if invalid_count == 0 else "FAIL"
    })

numeric_validation_summary = pd.DataFrame(
    numeric_validation_results
)

display(numeric_validation_summary)

,column,invalid_records,status
0,age,0,PASS
1,los_days,0,PASS
2,n_secondary_dx,0,PASS
3,n_medications,0,PASS
4,note_word_count,0,PASS
5,vs_spo2_pct,0,PASS
6,vs_height_cm,0,PASS
7,vs_weight_kg,0,PASS
8,vs_bmi,0,PASS


In [25]:
print(
    "Total numeric validation failures:",
    numeric_validation_summary["invalid_records"].sum()
)

Total numeric validation failures: 0


In [26]:
# ### 9. Categorical Data Profiling

# Before defining categorical validation rules, we inspect the distinct values
# present in important categorical columns.

# This allows the validation rules to be based on the actual source dataset
# rather than assuming values that may not exist in the source.

In [27]:
categorical_columns = [
    "encounter_type",
    "sex",
    "insurance_type",
    "facility_type",
    "age_band",
    "note_type"
]

for column in categorical_columns:
    print(f"\n{'=' * 50}")
    print(f"{column}")
    print(f"{'=' * 50}")
    print(df[column].value_counts(dropna=False))


encounter_type
encounter_type
outpatient    2286
emergency      638
inpatient      612
telehealth     429
lab_only       212
Name: count, dtype: int64

sex
sex
male      2314
female    1803
other       60
Name: count, dtype: int64

insurance_type
insurance_type
Medicare         1720
Commercial       1520
Medicaid          514
Dual-eligible     256
Uninsured         167
Name: count, dtype: int64

facility_type
facility_type
FQHC           892
Urgent Care    846
Clinic         819
ASC            812
Hospital       808
Name: count, dtype: int64

age_band
age_band
35-49    875
18-34    804
75+      796
50-64    744
65-74    626
0-17     332
Name: count, dtype: int64

note_type
note_type
SOAP         3048
Discharge     612
Radiology     517
Name: count, dtype: int64


In [28]:
# ### 10. Categorical Validation

# The following categorical fields are validated against the distinct values
# identified during source-data profiling.

# This prevents unexpected categorical values from entering the warehouse.

# The validation is based on the values present in the source dataset.

In [29]:
allowed_categories = {
    "encounter_type": {
        "outpatient",
        "emergency",
        "inpatient",
        "telehealth",
        "lab_only"
    },
    "sex": {
        "male",
        "female",
        "other"
    },
    "insurance_type": {
        "Medicare",
        "Commercial",
        "Medicaid",
        "Dual-eligible",
        "Uninsured"
    },
    "facility_type": {
        "FQHC",
        "Urgent Care",
        "Clinic",
        "ASC",
        "Hospital"
    },
    "age_band": {
        "0-17",
        "18-34",
        "35-49",
        "50-64",
        "65-74",
        "75+"
    },
    "note_type": {
        "SOAP",
        "Discharge",
        "Radiology"
    }
}

In [30]:
categorical_validation_results = []

for column, allowed_values in allowed_categories.items():

    invalid_mask = ~df[column].isin(allowed_values)

    invalid_count = invalid_mask.sum()

    categorical_validation_results.append({
        "column": column,
        "invalid_records": int(invalid_count),
        "status": "PASS" if invalid_count == 0 else "FAIL"
    })

categorical_validation_summary = pd.DataFrame(
    categorical_validation_results
)

display(categorical_validation_summary)

,column,invalid_records,status
0,encounter_type,0,PASS
1,sex,0,PASS
2,insurance_type,0,PASS
3,facility_type,0,PASS
4,age_band,0,PASS
5,note_type,0,PASS


In [31]:
print(
    "Total categorical validation failures:",
    categorical_validation_summary["invalid_records"].sum()
)

Total categorical validation failures: 0


In [32]:
# ### 11. Final Data Quality Report

# This section combines the results from all validation checks into a single
# data-quality report.

# The report provides a consolidated view of the ingestion quality before
# the dataset proceeds to transformation and warehouse loading.

In [33]:
# Combine all validation results

final_validation_summary = pd.concat(
    [
        validation_summary,
        numeric_validation_summary.rename(
            columns={"column": "validation"}
        ),
        categorical_validation_summary.rename(
            columns={"column": "validation"}
        )
    ],
    ignore_index=True
)

display(final_validation_summary)

,validation,invalid_records,status
0,Required columns,0,PASS
1,Required values,0,PASS
2,Unique encounter IDs,0,PASS
3,Valid dates,0,PASS
4,age,0,PASS
5,los_days,0,PASS
6,n_secondary_dx,0,PASS
7,n_medications,0,PASS
8,note_word_count,0,PASS
9,vs_spo2_pct,0,PASS


In [34]:
total_failures = final_validation_summary["invalid_records"].sum()

print("Total validation failures:", total_failures)

if total_failures == 0:
    print("OVERALL STATUS: PASS")
else:
    print("OVERALL STATUS: FAIL")

Total validation failures: 0
OVERALL STATUS: PASS


In [35]:
final_report_path = "../data/processed/final_validation_report.csv"

final_validation_summary.to_csv(
    final_report_path,
    index=False
)

print(f"Final validation report saved to: {final_report_path}")

Final validation report saved to: ../data/processed/final_validation_report.csv


In [36]:
# ### 12. Processed Dataset Verification

# The transformed dataset is loaded again to verify that the transformation
# preserved the record count and removed only the intended empty source field.

In [37]:
processed_df = pd.read_csv(
    "../data/processed/ehr_encounters_clean.csv"
)

print("Processed shape:", processed_df.shape)
print("Processed columns:", len(processed_df.columns))

print(
    "Contains fhir_bundle_path:",
    "fhir_bundle_path" in processed_df.columns
)

Processed shape: (4177, 101)
Processed columns: 101
Contains fhir_bundle_path: False


In [38]:
processed_df = pd.read_csv(
    "../data/processed/ehr_encounters_clean.csv"
)

print("Processed shape:", processed_df.shape)

print(
    "Contains fhir_bundle_path:",
    "fhir_bundle_path" in processed_df.columns
)

print(
    "Duplicate encounter IDs:",
    processed_df["encounter_id"].duplicated().sum()
)

print(
    "Missing encounter IDs:",
    processed_df["encounter_id"].isna().sum()
)

print(
    "Missing patient IDs:",
    processed_df["patient_id"].isna().sum()
)

Processed shape: (4177, 101)
Contains fhir_bundle_path: False
Duplicate encounter IDs: 0
Missing encounter IDs: 0
Missing patient IDs: 0


In [39]:
print("Raw rows:", len(df))
print("Processed rows:", len(processed_df))

if len(df) == len(processed_df):
    print("PASS: No records were lost during transformation.")
else:
    print("FAIL: Record count changed.")

Raw rows: 4177
Processed rows: 4177
PASS: No records were lost during transformation.
